# ISMRM 2027 travelling-head figures

Five original analyzed sessions and three additional EPI runs from 12 October 2026. Pilots 01/05 and sessions 08–10 are excluded. Figure 2 always exports an animated absolute/difference reference cycle and static absolute maps-only PNG/JPG backups in Stanford 1 space. Run export_abstract_metrics.py to validate map statistics and regenerate all numerical tables.

Figures 1–3 include a dark-gray `Frame n / N` footer in every animated frame. Figure 1 alternates B₀/B₁⁺ correlation and MAPD while MP2RAGE stays fixed.


## Shared backend 1 — imports and paths

In [ ]:
from pathlib import Path
import sys, json, hashlib, platform
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from IPython.display import display

WORK_DIR = Path.cwd().resolve()
if not (WORK_DIR / 'ismrm_2027_figures.ipynb').exists():
    WORK_DIR = WORK_DIR / 'ismrm27'
if not (WORK_DIR / 'ismrm_2027_figures.ipynb').exists():
    raise RuntimeError('Start from the repository root or its ismrm27 folder.')
sys.path.insert(0, str(WORK_DIR))
from data_helpers import read_image, read_correlations, read_mapd, read_pose, read_epi, source_record, METHODS

from animation_helpers import number_frame, FRAME_COUNTER

DATA_ROOT = Path(r'C:\Users\hmattern\Documents\Data\TH_Impulse_local')
DERIVATIVES = DATA_ROOT / 'derivatives' / 'ths_impulse_code'
OUTPUT_DIR = WORK_DIR
DPI = 300
# Larger typography shared by Figures 1–4 (including colorbars and annotations).
plt.rcParams.update({'font.size': 12.5, 'axes.titlesize': 15,
                     'axes.labelsize': 12.5, 'xtick.labelsize': 12.5,
                     'ytick.labelsize': 12.5, 'legend.fontsize': 12.5,
                     'figure.titlesize': 18})
SESSIONS = ['02', '03', '04', '06', '07']
SITES = {'02':'Stanford', '03':'Stanford', '04':'Berkeley', '06':'Magdeburg', '07':'Magdeburg', 'new1':'Magdeburg', 'new2':'Magdeburg', 'new3':'Magdeburg'}
# Figure labels use within-site visit numbers; data retain original session IDs.
DISPLAY_LABELS = {'02':'Stanford 1', '03':'Stanford 2', '04':'Berkeley', '06':'Magdeburg 1', '07':'Magdeburg 2'}
COLOURS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#767676']
MARKERS = ['o', 's', '^', 'D', 'v']

def scan_folder(session):
    return DATA_ROOT / 'scan_data' / f'THS_ses{session}_{SITES[session].lower()}' / 'nii'


## Shared backend 2 — image subplots and site comparisons

In [ ]:
def image_grid(rows, columns, chart_rows=0):
    fig = plt.figure(figsize=(2.4*columns+1, 2.7*rows+3*chart_rows+1.1), layout='constrained')
    grid = fig.add_gridspec(rows+chart_rows, columns+1,
                           height_ratios=[1]*rows+[1.25]*chart_rows, width_ratios=[1]*columns+[.08])
    axes = np.array([[fig.add_subplot(grid[r,c]) for c in range(columns)] for r in range(rows)])
    return fig, grid, axes

def image_panel(ax, image, data, view='axial', centre=None, offset_mm=0,
                limits=(0,1), cmap='gray', placeholder=False):
    # True world-RAS planes, including oblique images; neurological L/R display.
    x,y,fixed = {'axial':(0,1,2), 'coronal':(0,2,1), 'sagittal':(1,2,0)}[view]
    centre = np.array(centre if centre is not None else
                     nib.affines.apply_affine(image.affine,(np.array(image.shape)-1)/2), dtype=float)
    centre[fixed] += offset_mm
    width,height = (220,220) if view=='sagittal' else (180,220)
    xx,yy = np.meshgrid(np.linspace(-width/2,width/2,width),np.linspace(-height/2,height/2,height))
    world = np.broadcast_to(centre,(*xx.shape,3)).copy()
    world[...,x] += xx; world[...,y] += yy
    voxels = nib.affines.apply_affine(np.linalg.inv(image.affine),world)
    plane = map_coordinates(data,np.moveaxis(voxels,-1,0),order=1,mode='constant',cval=np.nan)
    artist = ax.imshow(plane,origin='lower',extent=[-width/2,width/2,-height/2,height/2],
                       cmap=plt.get_cmap(cmap).with_extremes(bad='black'),
                       vmin=limits[0],vmax=limits[1],interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    if view in ('axial','coronal'):
        ax.text(.02,.02,'L',transform=ax.transAxes,color='white',fontsize=10.0)
        ax.text(.94,.02,'R',transform=ax.transAxes,color='white',fontsize=10.0)
    if placeholder:
        ax.text(.5,.04,'PLACEHOLDER',transform=ax.transAxes,ha='center',color='white',fontsize=11.2,
                bbox={'facecolor':'black','alpha':.65,'pad':2})
    return artist, centre.tolist()

def display_scale(data, valid=None, percentiles=(1,99)):
    valid = np.isfinite(data)&(data>0) if valid is None else valid&np.isfinite(data)
    lo,hi = np.percentile(data[valid],percentiles)
    if hi<=lo:
        raise ValueError('Degenerate display range')
    return np.clip((data-lo)/(hi-lo),0,1), [float(lo),float(hi)]

def shade_sites(ax, sessions):
    groups=[]
    for i,s in enumerate(sessions):
        if not groups or groups[-1][0]!=SITES[s]:
            groups.append((SITES[s],[]))
        groups[-1][1].append(i)
    for site,indices in groups:
        ax.axvspan(indices[0]-.5,indices[-1]+.5,color='#fff6e5' if site=='Berkeley' else '#f3f5f7',zorder=-2)
        if indices[0]:
            ax.axvline(indices[0]-.5,color='#cccccc',lw=.7)
    ax.set_xlim(-.5,len(sessions)-.5)
    ax.set_xticks(range(len(sessions)),[DISPLAY_LABELS[s] for s in sessions],fontsize=12.5)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',color='#dddddd',lw=.6)
    ax.axhline(0,color='#888888',lw=.6)
    return groups

def site_series(ax, sessions, values, label, colour, marker):
    ax.plot(range(len(sessions)),values,linestyle='none',marker=marker,color=colour,label=label,zorder=3)
    for site in dict.fromkeys(SITES[s] for s in sessions):
        indices=[i for i,s in enumerate(sessions) if SITES[s]==site]
        if len(indices)>1:
            ax.plot(indices,np.asarray(values)[indices],color=colour,lw=1.5,label='_within_site_connection')


## Shared backend 3 — saving figures

In [ ]:
def save_figure(fig, name, caption, sources):
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    fig.savefig(OUTPUT_DIR/f'{name}.png',dpi=DPI,facecolor='white')
    fig.savefig(OUTPUT_DIR/f'{name}.jpg',dpi=DPI,facecolor='white',pil_kwargs={'quality':95})
    (OUTPUT_DIR/f'{name}_caption.txt').write_text(caption+'\n',encoding='utf-8')
    def serialise(value):
        if isinstance(value,np.ndarray): return value.tolist()
        if isinstance(value,np.generic): return value.item()
        if isinstance(value,Path): return str(value)
        raise TypeError(type(value).__name__)
    provenance={'caption':caption,'session_labels':DISPLAY_LABELS,'sources':sources,'dpi':DPI,
                'software':{'python':platform.python_version(),'numpy':np.__version__,
                            'nibabel':nib.__version__,'matplotlib':__import__('matplotlib').__version__},
                'notebook':source_record(WORK_DIR/'ismrm_2027_figures.ipynb'),
                'data_helpers':source_record(WORK_DIR/'data_helpers.py')}
    (OUTPUT_DIR/f'{name}_provenance.json').write_text(json.dumps(provenance,indent=2,default=serialise),encoding='utf-8')
    display(fig)
    plt.close(fig)
    print(f'Saved {name}.png and .jpg in {OUTPUT_DIR}')


## Figure 1 — correlation / MAPD animation

The two-frame GIF alternates B₀/B₁⁺ spatial correlations and mean absolute percentage differences (MAPD) every 2 seconds. Both MP2RAGE correlation panels remain identical. MAPD comes from the supplied heatmap annotations, exported at their displayed one-decimal precision to `ismrm_2027_b0_b1_mapd.csv`. All 7×7 directional values are validated before selecting sessions 02, 03, 04, 06 and 07. Pilots 01/05 are retained in the source CSV and excluded from display. Separate B₀/B₁⁺ MAPD scales and the correlation scale are labelled explicitly. A dark-gray frame counter appears in the GIF footer; the PNG/JPG backup remains a correlation figure.


In [ ]:
import io, base64
from PIL import Image
from IPython.display import HTML

correlation_csv = WORK_DIR / 'ismrm_2027_b0_b1_mp2rage_correlations.csv'
mapd_csv = WORK_DIR / 'ismrm_2027_b0_b1_mapd.csv'
mapd_source = WORK_DIR / 'ismrm_2027_b0_b1_mapd_provenance.json'
matrices = read_correlations(correlation_csv, SESSIONS)
mapd_matrices = read_mapd(mapd_csv, SESSIONS)
titles = ['B₀ · rigid', 'B₁⁺ · rigid', 'MP2RAGE · rigid', 'MP2RAGE · nonlinear']
correlation_sources = [source_record(correlation_csv,selected_sessions=SESSIONS,selected_matrices=matrices)]
caption = ('Spatial correlations for B₀/B₁⁺ rigid and patient-specific MP2RAGE rigid/nonlinear alignment. '
    'Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2; pilots excluded. Original directional asymmetry retained. '
    'Nonlinear concordance does not establish geometric fidelity or quantitative T₁ equivalence.')

# Preserve the existing static correlation backup and its layout.
fig,axes = plt.subplots(2,2,figsize=(11,10),layout='constrained')
for ax,(name,matrix),title,letter in zip(axes.flat,matrices.items(),titles,'ABCD'):
    artist = ax.imshow(matrix,cmap='viridis',vmin=.75,vmax=1)
    labels=[DISPLAY_LABELS[s] for s in SESSIONS]
    ax.set_xticks(range(5),labels,rotation=35,ha='right',fontsize=11.2)
    ax.set_yticks(range(5),labels,fontsize=11.2)
    ax.set_xlabel('Fixed session'); ax.set_ylabel('Moving session'); ax.set_title(f'{letter}  {title}')
    for i in range(5):
        for j in range(5):
            ax.text(j,i,f'{matrix[i,j]:.2f}',ha='center',va='center',color='white' if matrix[i,j]<.88 else 'black')
    for boundary in (1.5,2.5):
        ax.axhline(boundary,color='white',lw=2); ax.axvline(boundary,color='white',lw=2)
fig.colorbar(artist,ax=list(axes.flat),label='Spatial correlation r',shrink=.75)
fig.suptitle('Field-map and structural image correlations')
save_figure(fig,'figure_1_correlations',caption,correlation_sources)

# Fixed geometry and persistent bottom artists keep MP2RAGE pixel-identical.
fig = plt.figure(figsize=(12.8,10))
axes = [fig.add_axes([x,y,.30,.34]) for y in (.585,.10) for x in (.10,.58)]
top_bars = [fig.add_axes([x,.62,.012,.27]) for x in (.414,.894)]
bottom_bar = fig.add_axes([.894,.135,.012,.27])
fig.suptitle('Field-map reproducibility and structural image correlations',y=.975)
artists,annotations = [],[]
for ax,(name,matrix),title,letter in zip(axes,matrices.items(),titles,'ABCD'):
    artist=ax.imshow(matrix,cmap='viridis',vmin=.75,vmax=1)
    artists.append(artist)
    labels=[DISPLAY_LABELS[s] for s in SESSIONS]
    ax.set_xticks(range(5),labels,rotation=35,ha='right',fontsize=11.2)
    ax.set_yticks(range(5),labels,fontsize=11.2)
    ax.set_xlabel('Fixed session');ax.set_ylabel('Moving session')
    ax.set_title(f'{letter}  {title}')
    texts=[]
    for i in range(5):
        for j in range(5):
            texts.append(ax.text(j,i,f'{matrix[i,j]:.2f}',ha='center',va='center',
                                 color='white' if matrix[i,j]<.88 else 'black'))
    annotations.append(texts)
    for boundary in (1.5,2.5):
        ax.axhline(boundary,color='white',lw=2);ax.axvline(boundary,color='white',lw=2)
fig.colorbar(artists[2],cax=bottom_bar,label='Spatial correlation r')
correlation_frames,correlation_frame_records=[],[]
for frame_index,mode in enumerate(('correlation','mapd'),1):
    for panel,name in enumerate(('B0_rigid','B1_rigid')):
        matrix=matrices[name] if mode=='correlation' else mapd_matrices[name]
        limits=(.75,1) if mode=='correlation' else ((0,80) if panel==0 else (0,14))
        cmap='viridis' if mode=='correlation' else 'magma'
        artists[panel].set_data(matrix);artists[panel].set_cmap(cmap);artists[panel].set_clim(*limits)
        axes[panel].set_title(f'{"AB"[panel]}  {titles[panel]} · {"correlation" if mode=="correlation" else "MAPD"}')
        for text,value in zip(annotations[panel],matrix.flat):
            text.set_text(f'{value:.2f}' if mode=='correlation' else f'{value:.1f}')
            text.set_color('white' if value<(.88 if mode=='correlation' else .6*limits[1]) else 'black')
        top_bars[panel].clear()
        fig.colorbar(artists[panel],cax=top_bars[panel],
                     label='Spatial correlation r' if mode=='correlation' else 'MAPD (%)')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=120,facecolor='white');buffer.seek(0)
    correlation_frames.append(number_frame(Image.open(buffer),frame_index,2))
    correlation_frame_records.append({'index':frame_index-1,'number':frame_index,'count':2,
                                     'fieldmap_metric':mode,'mp2rage_metric':'correlation','duration_ms':2000})
plt.close(fig)
contact=Image.new('RGB',(correlation_frames[0].width*2,correlation_frames[0].height),'white')
for i,img in enumerate(correlation_frames):contact.paste(img,(i*img.width,0))
contact.save(OUTPUT_DIR/'figure_1_correlations_contact_sheet.png')
palette=contact.quantize(colors=256,method=Image.Quantize.MEDIANCUT)
indexed=[img.quantize(palette=palette,dither=Image.Dither.NONE) for img in correlation_frames]
gif_path=OUTPUT_DIR/'figure_1_correlations.gif'
indexed[0].save(gif_path,save_all=True,append_images=indexed[1:],duration=2000,loop=0,disposal=2,optimize=False)
correlation_frames[1].save(OUTPUT_DIR/'figure_1_mapd_preview.png')
animation_caption=(caption+' B₀/B₁⁺ correlation and MAPD panels alternate every 2 seconds (two frames, 4-second infinite loop). '
    'Both MP2RAGE panels remain fixed. MAPD values are transcribed from supplied rigid-registration heatmaps at one-decimal '
    'precision, with directional asymmetry and nonzero self-comparisons retained. Separate scales: B₀ MAPD 0–80%, '
    'B₁⁺ MAPD 0–14%; all correlations 0.75–1.00. The source CSV retains pilots 01/05, excluded from display. '
    'The footer shows the one-based frame number and total count in dark gray. PNG/JPG backups show correlations only.')
(OUTPUT_DIR/'figure_1_correlations_animation_caption.txt').write_text(animation_caption+'\n',encoding='utf-8')
provenance={'caption':animation_caption,'frames':correlation_frame_records,'frame_counter':FRAME_COUNTER,
    'session_labels':DISPLAY_LABELS,'sources':[*correlation_sources,
        source_record(mapd_csv,selected_sessions=SESSIONS,selected_matrices=mapd_matrices),source_record(mapd_source)],
    'scales':{'correlation':[.75,1],'B0_mapd_percent':[0,80],'B1_mapd_percent':[0,14]},
    'notebook':source_record(WORK_DIR/'ismrm_2027_figures.ipynb'),
    'data_helpers':source_record(WORK_DIR/'data_helpers.py'),
    'animation_helpers':source_record(WORK_DIR/'animation_helpers.py')}
(OUTPUT_DIR/'figure_1_correlations_animation_provenance.json').write_text(json.dumps(provenance,indent=2,default=lambda x:x.tolist() if isinstance(x,np.ndarray) else x)+'\n',encoding='utf-8')
encoded=base64.b64encode(gif_path.read_bytes()).decode('ascii')
display(HTML(f'<img src="data:image/gif;base64,{encoded}" alt="Figure 1 alternating B0 and B1 correlations and MAPD" style="max-width:100%;height:auto">'))
print(f'Saved {gif_path.name}: two frames, 2 seconds each; MP2RAGE remains fixed.')


## Figure 2 — animated field maps and static maps-only backup

Run this cell after the three shared backend cells. It **always creates** `figure_2_b0_b1_shims.gif` and the 300-dpi absolute maps-only backup `figure_2_b0_b1_maps.png` / `.jpg`. The GIF alternates absolute maps and signed differences every 2 seconds, cycling references **Stanford 1 → Stanford 2 → Berkeley → Magdeburg 1 → Magdeburg 2** in a 20-second infinite loop. Each reference uses matching saved rigid registrations and transversal planes ±20 mm from its brain centroid. B₀ change is in Hz; B₁⁺ change is relative percent, `100 × (session − reference) / reference`. The GIF retains acquisition shim plots and labels the reference column Reference in difference frames. The static backup has only absolute maps in Stanford 1 space, without difference panels, Reference labels, or shim charts. All exports show only Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2.


In [ ]:
# Figure 2: always create the reference-cycle GIF and absolute maps-only PNG/JPG backup.
# Run the three shared backend cells first.
import io, base64
from PIL import Image
from IPython.display import HTML

animation_sessions = ['02', '03', '04', '06', '07']
animation_ids = {'02':'260601_THS_ses02', '03':'260602_THS_ses03',
             '04':'260602_THS_ses04', '06':'260618_THS_ses06', '07':'260618_THS_ses07'}
animation_root = DATA_ROOT / 'derivatives' / 'rigid_b0_b1_maps_2026-10-14'
animation_b0_dir = animation_root / 'reg-gre_b0map_4iso_sag_ND_e1__corr-gre_b0map_4iso_sag_ND_romeo_hz_masked'
animation_b1_dir = animation_root / 'reg-dzne_b1map_5iso_sag__corr-dzne_b1map_5iso_sag_B1Comb_masked'
animation_offsets_mm = (-20, 20)
animation_duration_ms = 2000
animation_dpi = 100
animation_b0_absolute_limits = (-250, 250)
animation_b1_absolute_limits = (0, 120)
animation_b0_difference_limits = (-100, 100)
animation_b1_difference_limits = (-50, 50)
animation_b1_reference_floor = 10.0  # % nominal; suppress unstable small denominators.
animation_name = 'figure_2_b0_b1_shims'
backup_name = 'figure_2_b0_b1_maps'

animation_pose_csv = WORK_DIR / 'ismrm_2027_b0_shim_and_brain_orientation.csv'
animation_pose = read_pose(animation_pose_csv, animation_sessions)
animation_shim = animation_pose[((animation_pose.session!='03')&(animation_pose.setting==1))|
                        ((animation_pose.session=='03')&(animation_pose.setting==2))].set_index('session').loc[animation_sessions]
animation_sources = [source_record(animation_pose_csv, session03_selected_series='15–18')]
for animation_session in animation_sessions:
    if animation_session == '04':
        continue
    animation_series = '20' if animation_session == '03' else '11'
    animation_sidecar = scan_folder(animation_session) / f'dzne_b1map_5iso_sag_B1Comb_{animation_series}.json'
    if 'rel. B1 [0.1 %]' not in json.loads(animation_sidecar.read_text()).get('ImageComments', ''):
        raise ValueError(f'Session {animation_session}: unexpected B1 scaling')
    animation_sources.append(source_record(animation_sidecar, scale_to_percent_nominal=0.1))

animation_shim.reset_index().to_csv(OUTPUT_DIR/'fieldmap_shim_values.csv',index=False)
animation_frames, animation_frame_records, animation_qc = [], [], []
animation_positions = None
for animation_reference in animation_sessions:
    animation_fixed_id = animation_ids[animation_reference]
    animation_mask_path = DATA_ROOT / 'derivatives' / 'MP2RAGE_brainmask' / f'mask_{animation_fixed_id}.nii.gz'
    animation_mask_image, animation_mask_data = read_image(animation_mask_path)
    animation_mask_data = (np.isfinite(animation_mask_data)&(animation_mask_data>0)).astype(np.uint8)
    if not animation_mask_data.any():
        raise ValueError(f'Reference {animation_reference}: empty brain mask')
    animation_centre = nib.affines.apply_affine(animation_mask_image.affine, np.argwhere(animation_mask_data).mean(axis=0))
    animation_sources.append(source_record(animation_mask_path, reference=animation_reference,
                                      centre_ras_mm=animation_centre.tolist(), use='reference brain display mask'))
    animation_maps = {}
    for animation_kind, animation_directory, animation_suffix in (
        ('b0', animation_b0_dir, 'HZ'), ('b1', animation_b1_dir, 'CORR')):
        animation_reference_path = animation_directory / f'reg_Rigid_mov_{animation_fixed_id}_to_fix_{animation_fixed_id}_{animation_suffix}_Warped.nii.gz'
        animation_reference_image, animation_reference_data = read_image(animation_reference_path)
        if animation_kind == 'b1':
            animation_reference_data = animation_reference_data*0.1
        animation_voxels = np.indices(animation_reference_image.shape, dtype=float).reshape(3, -1).T
        animation_world = nib.affines.apply_affine(animation_reference_image.affine, animation_voxels)
        animation_mask_voxels = nib.affines.apply_affine(np.linalg.inv(animation_mask_image.affine), animation_world)
        animation_brain = map_coordinates(animation_mask_data, animation_mask_voxels.T, order=0,
                                      mode='constant', cval=0).reshape(animation_reference_image.shape)>0
        for animation_session in animation_sessions:
            animation_moving_id = animation_ids[animation_session]
            animation_path = animation_directory / f'reg_Rigid_mov_{animation_moving_id}_to_fix_{animation_fixed_id}_{animation_suffix}_Warped.nii.gz'
            animation_image, animation_data = read_image(animation_path)
            if (animation_image.shape != animation_reference_image.shape or
                    not np.allclose(animation_image.affine, animation_reference_image.affine, atol=1e-4, rtol=0)):
                raise ValueError(f'{animation_kind} {animation_session} to {animation_reference}: reference grid mismatch')
            if animation_kind == 'b1':
                animation_data = animation_data*0.1
            animation_valid = animation_brain & np.isfinite(animation_reference_data) & np.isfinite(animation_data)
            if animation_kind == 'b1':
                animation_valid &= (animation_reference_data>=animation_b1_reference_floor)&(animation_data>0)
            if animation_valid.sum()<100:
                raise ValueError(f'{animation_kind} {animation_session} to {animation_reference}: insufficient valid overlap')
            animation_difference = np.full(animation_data.shape, np.nan, dtype=np.float32)
            if animation_kind == 'b0':
                animation_difference[animation_valid] = animation_data[animation_valid]-animation_reference_data[animation_valid]
                animation_diff_limits = animation_b0_difference_limits
            else:
                animation_difference[animation_valid] = 100*(animation_data[animation_valid]-animation_reference_data[animation_valid])/animation_reference_data[animation_valid]
                animation_diff_limits = animation_b1_difference_limits
            if animation_session == animation_reference and not np.allclose(animation_difference[animation_valid], 0):
                raise ValueError('Reference-minus-itself must be zero')
            animation_maps[animation_kind,animation_session] = (animation_image, np.where(animation_valid,animation_data,np.nan), animation_difference)
            animation_values = animation_difference[animation_valid]
            animation_qc.append({'reference':animation_reference, 'session':animation_session, 'modality':animation_kind,
                             'valid_voxels':int(animation_valid.sum()), 'units':'Hz' if animation_kind=='b0' else 'relative_percent',
                             'difference_p01':float(np.percentile(animation_values,1)),
                             'difference_median':float(np.median(animation_values)),
                             'difference_p99':float(np.percentile(animation_values,99)),
                             'outside_difference_scale_percent':float(100*np.mean((animation_values<animation_diff_limits[0])|(animation_values>animation_diff_limits[1])))})
            animation_sources.append(source_record(animation_path, modality=animation_kind, moving_session=animation_session,
                                              fixed_session=animation_reference, transform='Rigid'))


    # Always save the absolute maps-only backup in Stanford 1 space, before GIF encoding.
    # The backup contains no difference panels, Reference columns, or shim charts.
    if animation_reference == '02':
        backup_fig, backup_grid, backup_axes = image_grid(4, len(animation_sessions))
        backup_qc = []
        for backup_kind, backup_first_row, backup_limits, backup_cmap in (
            ('b0', 0, animation_b0_absolute_limits, 'RdBu_r'),
            ('b1', 2, animation_b1_absolute_limits, 'viridis')):
            for backup_col, backup_session in enumerate(animation_sessions):
                backup_image, backup_data, _ = animation_maps[backup_kind,backup_session]
                backup_values = backup_data[np.isfinite(backup_data)]
                backup_p01, backup_q1, backup_median, backup_q3, backup_p99 = np.percentile(backup_values,[1,25,50,75,99])
                backup_qc.append({'session':backup_session,'modality':backup_kind,'fixed_session':'02',
                                  'units':'Hz' if backup_kind=='b0' else 'percent_nominal',
                                  'valid_brain_voxels':len(backup_values),'p01':backup_p01,'q1':backup_q1,
                                  'median':backup_median,'q3':backup_q3,'p99':backup_p99,
                                  'display_min':backup_limits[0],'display_max':backup_limits[1],
                                  'outside_display_percent':100*np.mean((backup_values<backup_limits[0])|(backup_values>backup_limits[1]))})
                for backup_level, backup_offset in enumerate(animation_offsets_mm):
                    backup_row=backup_first_row+backup_level
                    backup_artist, _ = image_panel(backup_axes[backup_row,backup_col],backup_image,backup_data,
                                                  view='axial',centre=animation_centre,offset_mm=backup_offset,
                                                  limits=backup_limits,cmap=backup_cmap)
                    if backup_row==0:
                        backup_axes[backup_row,backup_col].set_title(DISPLAY_LABELS[backup_session],fontsize=13.8)
                    if backup_col==0:
                        backup_symbol='B₀' if backup_kind=='b0' else 'B₁⁺'
                        backup_axes[backup_row,backup_col].set_ylabel(f'{backup_symbol} · {backup_offset:+} mm')
            backup_cax=backup_fig.add_subplot(backup_grid[backup_first_row:backup_first_row+2,5])
            backup_fig.colorbar(backup_artist,cax=backup_cax,
                               label='B₀ (Hz)' if backup_kind=='b0' else 'B₁⁺ (% nominal)',
                               extend='both' if backup_kind=='b0' else 'max')
        backup_fig.suptitle('Absolute B₀ / B₁⁺ maps\nTransversal views · Stanford 1 space')
        pd.DataFrame(backup_qc).to_csv(OUTPUT_DIR/'figure_2_fieldmap_qc.csv',index=False)
        backup_sources=[record for record in animation_sources if record.get('path')!=str(animation_pose_csv.resolve())]
        save_figure(backup_fig,backup_name,
            'Absolute B₀ frequency-offset (Hz) and combined B₁⁺ (% nominal) maps for Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2, '
            'rigidly registered to Stanford 1. Transversal world-RAS planes are ±20 mm from the Stanford 1 brain-mask '
            'centroid. Shared scales: B₀ −250 to +250 Hz; B₁⁺ 0–120% nominal (stored values ×0.1). '
            'The reference brain mask and valid session/reference overlap define the display region; '
            'B₁⁺ reference values below 10% nominal are excluded. Berkeley uses the supplied reconstructed '
            'export with consistent numerical and visual scale; reconstruction-side unit metadata are absent. '
            'This maps-only PNG/JPG backup is always exported alongside the animated Figure 2.',backup_sources)

    for animation_mode in ('absolute','difference'):
        animation_fig, animation_grid, animation_axes = image_grid(4, len(animation_sessions), chart_rows=1)
        for animation_kind, animation_first_row in (('b0',0),('b1',2)):
            if animation_mode == 'absolute':
                animation_limits = animation_b0_absolute_limits if animation_kind=='b0' else animation_b1_absolute_limits
                animation_cmap = 'RdBu_r' if animation_kind=='b0' else 'viridis'
                animation_label = 'B₀ (Hz)' if animation_kind=='b0' else 'B₁⁺ (% nominal)'
            else:
                animation_limits = animation_b0_difference_limits if animation_kind=='b0' else animation_b1_difference_limits
                animation_cmap = 'RdBu_r'
                animation_label = 'ΔB₀ (Hz)' if animation_kind=='b0' else 'ΔB₁⁺ (% vs reference)'
            for animation_col, animation_session in enumerate(animation_sessions):
                animation_image, animation_absolute, animation_delta = animation_maps[animation_kind,animation_session]
                animation_display = animation_absolute if animation_mode=='absolute' else animation_delta
                for animation_level, animation_offset in enumerate(animation_offsets_mm):
                    animation_row = animation_first_row+animation_level
                    animation_ax = animation_axes[animation_row,animation_col]
                    animation_artist, _ = image_panel(animation_ax, animation_image, animation_display, view='axial',
                                                 centre=animation_centre, offset_mm=animation_offset,
                                                 limits=animation_limits, cmap=animation_cmap)
                    if animation_mode=='difference' and animation_session==animation_reference:
                        animation_artist.set_data(np.full(animation_artist.get_array().shape, np.nan))
                        animation_ax.text(.5,.5,'Reference',transform=animation_ax.transAxes,
                                      color='white',ha='center',va='center',fontsize=18.8)
                    if animation_row==0:
                        animation_tag = 'Reference' if animation_session==animation_reference else ' '
                        animation_ax.set_title(f'{DISPLAY_LABELS[animation_session]}\n{animation_tag}',fontsize=13.8)
                    if animation_col==0:
                        animation_symbol = ('B₀' if animation_kind=='b0' else 'B₁⁺')
                        if animation_mode=='difference': animation_symbol = 'Δ'+animation_symbol
                        animation_ax.set_ylabel(f'{animation_symbol} · {animation_offset:+} mm')
            animation_cax=animation_fig.add_subplot(animation_grid[animation_first_row:animation_first_row+2,5])
            animation_fig.colorbar(animation_artist,cax=animation_cax,label=animation_label,
                               extend='max' if animation_mode=='absolute' and animation_kind=='b1' else 'both')
        animation_bottom=animation_grid[4,:5].subgridspec(1,2)
        for animation_panel,(animation_columns,animation_labels,animation_title,animation_unit) in enumerate((
            ([f'shim_offset_{a}_dac' for a in 'xyz'],['X','Y','Z'],'First-order offsets','DAC counts'),
            ([f'shim_current_{i}_ma' for i in range(5)],['A20','A21','B21','A22','B22'],'Second-order currents','mA'))):
            animation_ax=animation_fig.add_subplot(animation_bottom[0,animation_panel]); shade_sites(animation_ax,animation_sessions)
            for animation_i,(animation_column,animation_legend) in enumerate(zip(animation_columns,animation_labels)):
                site_series(animation_ax,animation_sessions,animation_shim[animation_column].to_numpy(),animation_legend,COLOURS[animation_i],MARKERS[animation_i])
            animation_ax.set_title(animation_title);animation_ax.set_ylabel(animation_unit)
            animation_ax.legend(loc='upper center',bbox_to_anchor=(.5,-.13),ncol=3,frameon=False,fontsize=11.2)
        animation_mode_title = 'Absolute maps' if animation_mode=='absolute' else 'Signed change: session − reference'
        animation_fig.suptitle(f'{animation_mode_title}\nTransversal views · rigid reference: {DISPLAY_LABELS[animation_reference]}')
        # Freeze axes geometry to avoid layout jumps when labels/scales alternate.
        animation_fig.canvas.draw()
        if animation_positions is None:
            animation_positions=[a.get_position().frozen() for a in animation_fig.axes]
        animation_fig.set_layout_engine(None)
        for animation_ax,animation_position in zip(animation_fig.axes,animation_positions):
            animation_ax.set_position(animation_position)
        animation_buffer=io.BytesIO()
        animation_fig.savefig(animation_buffer,format='png',dpi=animation_dpi,facecolor='white')
        animation_buffer.seek(0)
        animation_frames.append(number_frame(Image.open(animation_buffer),len(animation_frames)+1,2*len(animation_sessions)))
        animation_frame_records.append({'index':len(animation_frames)-1,'number':len(animation_frames),'count':2*len(animation_sessions),'reference':animation_reference,
                                    'mode':animation_mode,'duration_ms':animation_duration_ms,
                                    'slice_centre_ras_mm':animation_centre.tolist(),
                                    'slice_offsets_mm':list(animation_offsets_mm)})
        plt.close(animation_fig)
    print(f'Animation frames ready: reference {animation_reference}', flush=True)

# One palette for every frame prevents GIF palette flicker.
animation_thumbs=[]
for animation_frame in animation_frames:
    animation_thumb=animation_frame.copy();animation_thumb.thumbnail((390,450),Image.Resampling.LANCZOS)
    animation_thumbs.append(animation_thumb)
animation_tw,animation_th=animation_thumbs[0].size
animation_contact=Image.new('RGB',(5*animation_tw,2*animation_th),'white')
for animation_i,animation_thumb in enumerate(animation_thumbs):
    animation_contact.paste(animation_thumb,((animation_i//2)*animation_tw,(animation_i%2)*animation_th))
animation_contact.save(OUTPUT_DIR/f'{animation_name}_contact_sheet.png')
animation_palette=animation_contact.quantize(colors=256,method=Image.Quantize.MEDIANCUT)
animation_indexed=[f.quantize(palette=animation_palette,dither=Image.Dither.NONE) for f in animation_frames]
animation_gif_path=OUTPUT_DIR/f'{animation_name}.gif'
animation_indexed[0].save(animation_gif_path,save_all=True,append_images=animation_indexed[1:],
                      duration=animation_duration_ms,loop=0,disposal=2,optimize=False)
# Full-size difference preview supports inspection without chasing animation frames.
animation_frames[1].save(OUTPUT_DIR/f'{animation_name}_difference_ses02.png')
pd.DataFrame(animation_qc).to_csv(OUTPUT_DIR/f'{animation_name}_qc.csv',index=False)
animation_caption = (
    'Figure 2 GIF: absolute B₀/B₁⁺ maps and signed differences alternate every 2 seconds, cycling '
    'through references Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2 (10 frames, 20 seconds, infinite loop). '
    'Each reference uses the corresponding saved moving-to-fixed rigid registrations and its own '
    'brain-mask centroid; planes are transversal world-RAS at ±20 mm. The physical reference space '
    'changes when the reference changes. Absolute and difference frames within a reference share '
    'the same valid overlap mask. B₀ difference = session − reference (Hz); B₁⁺ change = '
    '100 × (session − reference) / reference, a relative percent change, not percentage points. '
    'B₁⁺ reference values below 10% nominal are excluded. Absolute scales: B₀ ±250 Hz, B₁⁺ 0–120% nominal; '
    'difference scales: B₀ ±100 Hz, B₁⁺ ±50% relative. Positive differences are red, negative blue. '
    'The reference column is labelled Reference in difference frames. Acquisition shim plots are unchanged. '
    'B₁⁺ stored values ×0.1; Berkeley uses the supplied reconstructed export with consistent scale; '
    'its reconstruction-side unit metadata are absent. A maps-only absolute PNG/JPG backup in Stanford 1 space is always created by the same cell.')
(OUTPUT_DIR/f'{animation_name}_caption.txt').write_text(animation_caption+'\n',encoding='utf-8')
animation_provenance={'frame_counter':FRAME_COUNTER,'animation_helpers':source_record(WORK_DIR/'animation_helpers.py'),'caption':animation_caption,'session_labels':DISPLAY_LABELS,'reference_order':animation_sessions,'frames':animation_frame_records,
                  'sources':animation_sources,'b1_reference_floor_percent_nominal':animation_b1_reference_floor,
                  'difference_formulas':{'b0':'moving_hz - reference_hz',
                                         'b1':'100 * (moving_percent_nominal - reference_percent_nominal) / reference_percent_nominal'},
                  'scales':{'b0_absolute_hz':animation_b0_absolute_limits,'b1_absolute_percent_nominal':animation_b1_absolute_limits,
                            'b0_difference_hz':animation_b0_difference_limits,'b1_relative_change_percent':animation_b1_difference_limits},
                  'notebook':source_record(WORK_DIR/'ismrm_2027_figures.ipynb'),
                  'software':{'python':platform.python_version(),'numpy':np.__version__,'nibabel':nib.__version__,
                              'matplotlib':__import__('matplotlib').__version__,'Pillow':__import__('PIL').__version__}}
(OUTPUT_DIR/f'{animation_name}_provenance.json').write_text(json.dumps(animation_provenance,indent=2)+'\n',encoding='utf-8')
animation_encoded=base64.b64encode(animation_gif_path.read_bytes()).decode('ascii')
display(HTML(f'<img src="data:image/gif;base64,{animation_encoded}" alt="Figure 2 animated maps and reference differences" style="max-width:100%;height:auto">'))
print(f'Saved {animation_gif_path.name}: {len(animation_frames)} frames, 2 seconds each, 20-second infinite loop.')
print(f'Static backup: {backup_name}.png and .jpg (absolute maps only).')


## Figure 3 — Rigid / SyN MP2RAGE in Stanford 1 space

This cell always creates a **GIF alternating Rigid and SyN every 2 seconds**, plus a static **Rigid PNG/JPG** backup. Only Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2 are shown, on identical transversal, coronal and sagittal planes in Stanford 1 space. Each session uses its Rigid-image brain 1st/99th percentile display range in both frames. Lower pose plots retain original native brain-centroid shifts and PCA rotations, rather than registration parameters.


In [ ]:
# Always export a 2-second Rigid/SyN GIF and static Rigid PNG/JPG backup.
import io, base64
from PIL import Image
from IPython.display import HTML

frame = 'RAS'  # Native pose plot coordinates; image planes always use world RAS.
mp_sessions = ['02','03','04','06','07']
mp_ids = {'02':'260601_THS_ses02','03':'260602_THS_ses03','04':'260602_THS_ses04',
          '06':'260618_THS_ses06','07':'260618_THS_ses07'}
mp_root = DATA_ROOT/'derivatives'/'mp2rage_0p7iso_patientSpecific_UNI-DEN_ND_defaced'
mp_fixed_id = mp_ids['02']
mp_mask_path = DATA_ROOT/'derivatives'/'MP2RAGE_brainmask'/f'mask_{mp_fixed_id}.nii.gz'
mp_mask_image, mp_mask_values = read_image(mp_mask_path)
mp_brain = np.isfinite(mp_mask_values)&(mp_mask_values>0)
if not mp_brain.any(): raise ValueError('Empty Stanford 1 brain mask')
mp_centre = nib.affines.apply_affine(mp_mask_image.affine,np.argwhere(mp_brain).mean(axis=0))
pose_csv = WORK_DIR/'ismrm_2027_b0_shim_and_brain_orientation.csv'
pose = read_pose(pose_csv,mp_sessions).drop_duplicates('session').set_index('session').loc[mp_sessions]
pose.reset_index().to_csv(OUTPUT_DIR/'mp2rage_pose_values.csv',index=False)
mp_sources = [source_record(mp_mask_path,fixed_session='02',slice_centre_ras_mm=mp_centre.tolist(),
                            use='common slice centre and brain region for display percentiles'),
              source_record(pose_csv,pose_frame=frame,pose_meaning='original native brain pose; not registration parameters')]
mp_scales, mp_qc, mp_frames, mp_positions = {}, [], [], None
mp_frame_duration_ms = 2000
mp_name = 'figure_3_mp2rage_pose'
mp_static_name = 'figure_3_mp2rage_pose_preview'
for mp_transform in ('Rigid','SyN'):
    fig,grid,axes = image_grid(3,len(mp_sessions),chart_rows=2)
    for col,session in enumerate(mp_sessions):
        mp_path = mp_root/f'reg_{mp_transform}_mov_{mp_ids[session]}_to_fix_{mp_fixed_id}_CORR_Warped.nii.gz'
        image,data = read_image(mp_path)
        if image.shape!=mp_mask_image.shape or not np.allclose(image.affine,mp_mask_image.affine,atol=1e-4,rtol=0):
            raise ValueError(f'{mp_transform} session {session}: grid differs from Stanford 1')
        valid = mp_brain&np.isfinite(data)&(data>0)
        if valid.sum()<100: raise ValueError(f'{mp_transform} session {session}: insufficient valid brain voxels')
        if mp_transform=='Rigid':
            mp_scales[session] = np.percentile(data[valid],[1,99]).tolist()
        lo,hi = mp_scales[session]
        if hi<=lo: raise ValueError('Degenerate MP2RAGE display range')
        scaled = np.clip((data-lo)/(hi-lo),0,1)
        positions=[]
        for row,view in enumerate(('axial','coronal','sagittal')):
            _,centre = image_panel(axes[row,col],image,scaled,view=view,centre=mp_centre)
            positions.append(centre)
            if row==0: axes[row,col].set_title(DISPLAY_LABELS[session],fontsize=13.8)
            if col==0: axes[row,col].set_ylabel('Transversal' if view=='axial' else view.capitalize())
        mp_sources.append(source_record(mp_path,session=session,transform=mp_transform,fixed_session='02',
                         display_percentiles=mp_scales[session],scaling_basis='session Rigid image: brain 1st/99th percentiles; reused for SyN',
                         slice_centres_ras_mm=positions))
        mp_qc.append({'session':session,'transform':mp_transform,'fixed_session':'02','brain_voxels':int(valid.sum()),
                      'display_low':lo,'display_high':hi,'brain_median':float(np.median(data[valid])),
                      'outside_display_percent':float(100*np.mean((data[valid]<lo)|(data[valid]>hi)))})
        print(f'Figure 3: {mp_transform} session {session}',flush=True)
        del data,scaled,valid
    cax=fig.add_subplot(grid[:3,5]);fig.colorbar(axes[0,0].images[0],cax=cax,label='Display intensity (0–1)')
    labels=['X (right)','Y (anterior)','Z (superior)'] if frame=='RAS' else ['X (left)','Y (posterior)','Z (superior)']
    for row,(kind,suffix,title,unit) in enumerate((
        ('translation','mm','Native brain-centroid shift from image-volume centre','Shift (mm)'),
        ('rotation','deg','Original native brain-mask orientation · fixed-axis xyz','Rotation (degrees)'))):
        ax=fig.add_subplot(grid[3+row,:5]);shade_sites(ax,mp_sessions)
        for i,(axis,label) in enumerate(zip('xyz',labels)):
            site_series(ax,mp_sessions,pose[f'{kind}_{frame.lower()}_{axis}_{suffix}'].to_numpy(),label,COLOURS[i],MARKERS[i])
        ax.set_title(f'{title} · {frame}');ax.set_ylabel(unit)
        ax.legend(loc='upper center',bbox_to_anchor=(.5,-.13),ncol=3,frameon=False)
    fig.suptitle(f'{mp_transform} registration\nAll images co-registered to Stanford 1')
    fig.canvas.draw()
    if mp_positions is None: mp_positions=[ax.get_position().frozen() for ax in fig.axes]
    fig.set_layout_engine(None)
    for ax,position in zip(fig.axes,mp_positions):ax.set_position(position)
    # Render the animated frame before save_figure closes the Rigid backup figure.
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,facecolor='white');buffer.seek(0)
    mp_frames.append(number_frame(Image.open(buffer),len(mp_frames)+1,2))
    if mp_transform=='Rigid':
        save_figure(fig,mp_static_name,
            f'Patient-specific denoised MP2RAGE UNI images for Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2, rigidly co-registered to Stanford 1. '
            f'Transversal, coronal and sagittal world-RAS planes pass through the Stanford 1 brain-mask centroid. '
            f'Each session uses its Rigid image brain 1st–99th percentile range for display; these ranges are also reused for SyN in the GIF. '
            f'Display scaling is distinct from WM metric normalization. Lower plots retain original native brain-centroid shifts '
            f'from native volume centres and PCA fixed-axis xyz rotations in {frame}, not registration transforms. '
            f'This static Rigid PNG/JPG backup is always exported with the animated Figure 3.',mp_sources.copy())
    else:plt.close(fig)
# One shared palette, identical layout and per-session display scaling across frames.
mp_contact=Image.new('RGB',(mp_frames[0].width*2,mp_frames[0].height),'white')
for i,img in enumerate(mp_frames):mp_contact.paste(img,(i*img.width,0))
mp_contact.save(OUTPUT_DIR/f'{mp_name}_contact_sheet.png')
mp_palette=mp_contact.quantize(colors=256,method=Image.Quantize.MEDIANCUT)
mp_indexed=[img.quantize(palette=mp_palette,dither=Image.Dither.NONE) for img in mp_frames]
mp_gif_path=OUTPUT_DIR/f'{mp_name}.gif'
mp_indexed[0].save(mp_gif_path,save_all=True,append_images=mp_indexed[1:],duration=mp_frame_duration_ms,
                   loop=0,disposal=2,optimize=False)
pd.DataFrame(mp_qc).to_csv(OUTPUT_DIR/f'{mp_name}_registration_qc.csv',index=False)
mp_caption = ('Figure 3 alternates Rigid and SyN (nonlinear) registrations every 2 seconds, in an infinite 4-second loop. '
    'All images from Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2 are co-registered to Stanford 1. Identical transversal, coronal and sagittal '
    'world-RAS planes pass through the Stanford 1 brain centroid. Each session uses a fixed display range derived from its '
    'Rigid-image brain 1st/99th percentiles, reused for its SyN frame; this is display scaling, not quantitative T1 or WM metric normalization. '
    f'Lower plots describe original native brain-centroid shifts and PCA rotations in {frame}; they remain unchanged between frames. '
    'The Rigid PNG/JPG backup is always created.')
(OUTPUT_DIR/f'{mp_name}_caption.txt').write_text(mp_caption+'\n',encoding='utf-8')
mp_provenance={'frame_counter':FRAME_COUNTER,'animation_helpers':source_record(WORK_DIR/'animation_helpers.py'),'caption':mp_caption,'session_labels':DISPLAY_LABELS,'sources':mp_sources,'reference_session':'02','transforms':['Rigid','SyN'],
               'frames':[{'index':i,'number':i+1,'count':2,'transform':t,'duration_ms':mp_frame_duration_ms} for i,t in enumerate(('Rigid','SyN'))],
               'slice_centre_ras_mm':mp_centre.tolist(),'display_ranges_by_session':mp_scales,
               'notebook':source_record(WORK_DIR/'ismrm_2027_figures.ipynb'),
               'data_helpers':source_record(WORK_DIR/'data_helpers.py'),
               'software':{'python':platform.python_version(),'numpy':np.__version__,'nibabel':nib.__version__,
                           'matplotlib':__import__('matplotlib').__version__,'Pillow':__import__('PIL').__version__}}
(OUTPUT_DIR/f'{mp_name}_provenance.json').write_text(json.dumps(mp_provenance,indent=2)+'\n',encoding='utf-8')
mp_encoded=base64.b64encode(mp_gif_path.read_bytes()).decode('ascii')
display(HTML(f'<img src="data:image/gif;base64,{mp_encoded}" alt="Figure 3 Rigid and SyN comparison" style="max-width:100%;height:auto">'))
print(f'Saved {mp_gif_path.name}: Rigid ↔ SyN, 2 seconds per frame; static Rigid PNG/JPG also saved.')


## Figure 4 — 2D EPI image quality and temporal stability

Native maps for five original sessions and representative October run 2. Quantitative panels and validated CSV exports include all three October runs with unchanged settings. Run 1 was unintentional preheating; runs 2–3 are summarized separately. Whiskers show within-run voxel quartiles. Processing gains include interpolation.


In [ ]:
epi_csv = WORK_DIR / 'ismrm_2027_ep2d_summary.csv'
export_manifest = DERIVATIVES / 'epi_qc' / 'abstract_export' / 'export_manifest.json'
runs = ['02','03','04','06','07','new1','new2','new3']
image_runs = ['02','03','04','06','07','new2']
tsnr_limits = (0,60)
image_centres = {}  # Optional native RAS-mm centre overrides, keyed by run.
table,maps,map_sources = read_epi(epi_csv,export_manifest,runs,DATA_ROOT,DERIVATIVES)
from export_abstract_metrics import export_epi
table = export_epi(table)
fig,grid,axes=image_grid(3,6,chart_rows=1)
sources=[source_record(epi_csv),source_record(export_manifest),*map_sources]
for col,run in enumerate(image_runs):
    positions=[]
    for r,kind in enumerate(('mean','raw_tsnr','motion_corrected_detrended_tsnr')):
        image,data=maps[run]['images'][kind]
        if kind=='mean': data,scale=display_scale(data,np.isfinite(data))
        artist,centre=image_panel(axes[r,col],image,data,centre=image_centres.get(run,maps[run]['centre']),
                                 limits=(0,1) if kind=='mean' else tsnr_limits,cmap='gray' if kind=='mean' else 'viridis')
        positions.append(centre)
        if r==0:
            title=DISPLAY_LABELS[run] if run!='new2' else 'Magdeburg after\nrecalibration, run2'
            axes[r,col].set_title(title,fontsize=13.8,color='black')
        if col==0: axes[r,col].set_ylabel(['A  Mean EPI\n(display scaled)','B  Raw tSNR','C  Motion corrected\n+ detrended tSNR'][r])
        if run=='new2':
            for spine in axes[r,col].spines.values(): spine.set_color('#18764b'); spine.set_linewidth(2)
    sources.append({'run':run,'slice_centres_ras_mm':positions})
cax=fig.add_subplot(grid[1:3,6]); fig.colorbar(artist,cax=cax,label='tSNR · shared scale')
ax=fig.add_subplot(grid[3,:6])
for i,run in enumerate(runs):
    ax.axvspan(i-.5,i+.5,color='#e7f4ec' if run.startswith('new') else '#fff6e5' if run=='04' else '#f3f5f7',zorder=-2)
    if i and (SITES[runs[i-1]]!=SITES[run] or run=='new1'): ax.axvline(i-.5,color='#b9c6bf',ls='--',lw=.8)
for j,(metric,label) in enumerate(zip(METHODS,['Raw','Linear detrend','Motion correction + linear detrend'])):
    median=table[f'{metric}_median'].to_numpy()
    q1,q3=table[f'{metric}_q1'].to_numpy(),table[f'{metric}_q3'].to_numpy()
    ax.errorbar(np.arange(len(runs))+(j-1)*.18,median,yerr=[median-q1,q3-median],fmt=MARKERS[j],color=COLOURS[j],label=label,capsize=3,ms=5)
ax.set_xlim(-.5,len(runs)-.5); ax.set_ylim(0,50); ax.grid(axis='y',color='#dddddd',lw=.6)
ax.spines[['top','right']].set_visible(False)
ax.set_xticks(range(len(runs)),[DISPLAY_LABELS[s] if not s.startswith('new') else f'run {s[-1]}'+('\nMagdeburg after recalibration' if s=='new2' else '') for s in runs],fontsize=11.2)
ax.set_ylabel('tSNR median and voxel Q1–Q3')
ax.set_title('D  Processing effects within each run · whiskers show spatial voxel dispersion')
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.17),ncol=3,frameon=False)
fig.suptitle('2D EPI image quality and temporal stability\nNative axial views, not coregistered · first 10 excluded · 370 retained volumes at TR 1 s')
save_figure(fig,'figure_4_ep2d_temporal_stability',
    'Native mean EPI, raw tSNR and motion-corrected/linearly detrended tSNR for Stanford 1, Stanford 2, Berkeley, Magdeburg 1 and Magdeburg 2 and Magdeburg Add-on (12 October 2026, source new2) after Frequalizer recalibration and the first repeated run, which served as unintentional preheating. Quantitative points include all three unchanged-protocol October runs, in acquisition order. 380 acquired/370 retained volumes, TR 1 s, TE 24 ms, population SD. Mean EPI is independently display-scaled; tSNR rows share a scale. Each run uses its same native brain mask; map quartiles/counts were checked against the CSV. Points show medians, whiskers within-run voxel Q1–Q3. Across-run statistics separate the first October run from subsequent runs 2–3; n=1 has no across-run IQR. Processing gains include interpolation; recalibration and preheating effects cannot be separated. Repeated runs are not independent participants.',sources)
display(pd.read_csv(OUTPUT_DIR/'epi_across_run_summary.csv'))
